## 


Démonstration du problème d'un "nettoyage" après un max-pooling, afin d'assurer sa restitution par un unpooling.

L'opération de *max-pooling* consiste à réduire la taille d'un tenseur $t1$ en ne conservant que la valeur maximale dans une fenêtre glissante. Cette réduction en $t2$ n'est pas réversible car seules les valeurs maximales sont conservées, les autres étant perdues. Toutefois, bien qu'il ne soit pas possible de reconstruire le tenseur d'origine à partir du tenseur réduit, on peut néanmoins tenter de reconstituer une approximation $t1'$ de $t1$ grâce à une opération de *max-unpooling* (définie avec les mêmes paramètres que le *max-pooling*), qui ne fait que réinsérer les valeurs de $t2$ dans $t1'$ (de dimensions identiques à $t1$) dans les positions fournies par le *max-pooling*, correspondant aux indices de localisation des maximums ("*switches*") dans $t1$.

Dans la procédure de déconvolution pour visualiser la structure d'une image ayant excité un neurone surveillé, si celui-ci se trouve dans la couche cachée en sortie du module de *max-pooling*, l'opération de *max-unpooling* est réalisée sur une matrice nulle sauf à la position du neurone pour laquelle la valeur d'activation est concervée (matrice obtenue par une opération de "nettoyage" de la couche cachée), avec l'utilisation des indices de localisation des maximums fournis par le module de *max-pooling* ont exploités.

Cette opération de "nettoyage" comporte une faille. En effet, si l'indice de localisation "*switches*" associé au neurone surveillé est présent plusieurs fois, et que celui-ci n'est pas la dernière occurrence présente dans la liste des indices selon le sens de parcours de cette liste par l'algorithme de *max-unpooling*, alors il peut y avoir écrasement de la valeur maximum dans le tenseur reconstruit par 0.

Plus généralement, on peut étendre ce problème à l'ensemble des étapes de *max-unpooling* de la déconvolution, où l'on peut se retrouver avec des valeurs nulles dans le tenseur reconstruit à la place de valeur maximum d'origine.

Voici un exemple illustrant ce problème en utilisant la librairie PyTorch (`torch`). Nous allons créer un tenseur $t1$ de taille $(5, 5)$, puis effectuer un *max-pooling* avec une taille de fenêtre de $(3, 3)$, d'un pas de $2$ et de padding $0$, pour obtenir $t2$. Après avoir "nettoyé" $t2$ en concervant la valeur d'activation du neurone surveillé, nous allons y appliquer un *max-unpooling* en utilisant les indices de localisation des maximums fournis par le *max-pooling*, et constater la présence du maximum de $t1$.

In [39]:
import numpy as np
import torch
import torch.nn as nn

Créons un tenseur 2D aléatoire pour lequel nous allons placer une valeur maximale à une position donnée.

In [40]:
# Création d'un tenseur de dimension (5, 5)
np.random.seed(71)
a = np.arange(0, 5*5)
np.random.shuffle(a)
t1 = torch.tensor(a, dtype=torch.float32).reshape((5, 5))
# Assurer que le maximum soit en position (1, 2)
position_max = (1, 2)
t1[position_max] = t1.max() + 1
print("Le tenseur t1 vaut :")
print(t1)
print(f"Il a pour maximum : {t1.max().item()} en position {position_max} (indexation en base 0)")

Le tenseur t1 vaut :
tensor([[23.,  5.,  2., 17.,  4.],
        [12., 15., 25., 10., 22.],
        [20.,  0.,  9.,  3., 14.],
        [19.,  6., 16.,  1., 18.],
        [21.,  8., 24., 13., 11.]])
Il a pour maximum : 25.0 en position (1, 2) (indexation en base 0)


Appliquons un *max-pooling* du type de ceux présent dans un réseau ConvNet sur ce tenseur :

In [41]:
# Instanciation d'une opération de maxpooling 2D tel que construite dans un réseau ConvNet
maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=0, return_indices=True)
# On rend les dimensions de t1 compatibles avec l'opération de maxpooling
t1_reshaped = t1.unsqueeze(0)
t2, indices  = maxpool(t1_reshaped)

print("Le tenseur après maxpool t2 vaut :")
print(t2.squeeze())
print(f"Et les indices de localisation sont (numérotation linéaire entre 0 et {t1.numel()-1}):")
print(indices.squeeze())

Le tenseur après maxpool t2 vaut :
tensor([[25., 25.],
        [24., 24.]])
Et les indices de localisation sont (numérotation linéaire entre 0 et 24):
tensor([[ 7,  7],
        [22, 22]])


On remarque que la maximum est présent au moins deux fois dans le tenseur réduit. Si nous surveillons le neurone (0, 1), et opérons le "nettoyage" de $t2$, on obtient un tenseur de la forme suivante :

In [42]:
t2_cleaned = torch.zeros_like(t2)
t2_cleaned[0, 0, 0] = t1.max().item()
print("Le tenseur t2 nettoyé en préservant (0, 0) vaut :")
print(t2_cleaned.squeeze(0).squeeze(0))

Le tenseur t2 nettoyé en préservant (0, 0) vaut :
tensor([[25.,  0.],
        [ 0.,  0.]])


Opérons maintenant un *max-unpooling*, "réciproque" du *max-pooling* appliqué sur $t1$, sur le tenseur réduit $t2$, en utilisant les indices de localisation des maximums fournis en appliquant *max-pooling*:

In [43]:
maxunpool = nn.MaxUnpool2d(
    kernel_size=maxpool.kernel_size,
    stride=maxpool.stride,
    padding=maxpool.padding
)

t1_rebuilt = maxunpool(t2_cleaned, indices)
print("Le tenseur reconstruit t1_rebuilt vaut :")
print(t1_rebuilt.squeeze(0))

Le tenseur reconstruit t1_rebuilt vaut :
tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


On constate l'absence du maximum. Si avait surveillé le neurone (0, 1), et que l'on avait appliqué le "nettoyage" de $t2$, on aurait obtenu un tenseur de la forme suivante, dont le résultat en appliquant le *max-unpooling* est :

In [44]:
t2_cleaned = torch.zeros_like(t2)
t2_cleaned[0, 0, 1] = t2.max().item()
print("Le tenseur t2 nettoyé en préservant (0, 1) vaut :")
print(t2_cleaned)

t1_rebuilt = maxunpool(t2_cleaned, indices)
print("Le tenseur reconstruit t1_rebuilt vaut :")
print(t1_rebuilt.squeeze())

Le tenseur t2 nettoyé en préservant (0, 1) vaut :
tensor([[[ 0., 25.],
         [ 0.,  0.]]])
Le tenseur reconstruit t1_rebuilt vaut :
tensor([[ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0., 25.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.]])


Ici, on constate que le maximum est bien présent à la position qu'il avait dans le tenseur d'origine.

Que se passe-t'il quand le neurone (0, 0) est surveillé ? Le max-unpooling a suivi les différentes étapes :

1. $t1'$ est initialisé à une matrice nulle de la même taille que $t1$ :
$\begin{pmatrix}
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0
\end{pmatrix}$

2. $t1'$ est mis à jour selon la valeur de $t2$ à la première position $(0, 0)", à savoir "7 \rightarrow (1, 2)$ présente dans les indices de localisation, c'est à dire $25$, ce qui donne :

$t1' = \begin{pmatrix}
\begin{bmatrix}
0 & 0 & 0 \\
0 & 0 & \mathbf{25} \\
0 & 0 & 0
\end{bmatrix} & \begin{array}{cc}
0 & 0 \\
0 & 0 \\
0 & 0
\end{array} \\
\begin{array}{ll}
0 & 0 & 0 \\
0 & 0 & 0
\end{array} & \begin{array}{cc}
0 & 0 \\
0 & 0
\end{array}
\end{pmatrix}$

2. $t1'$ est mis à jour selon la valeur de $t2$ à la deuxième position $(0, 1)", à savoir "7 \rightarrow (1, 2)$ présente dans les indices de localisation, c'est à dire $0$, ce qui donne :

$t1' = \begin{pmatrix}
\begin{array}{cc}
0 & 0 \\
0 & 0 \\
0 & 0
\end{array} &
\begin{bmatrix}
0 & 0 & 0 \\
\mathbf{0} & 0 & 0 \\
0 & 0 & 0
\end{bmatrix} \\
\begin{array}{cc}
0 & 0 \\
0 & 0
\end{array} &
\begin{array}{ll}
0 & 0 & 0 \\
0 & 0 & 0
\end{array}
\end{pmatrix}$

La valeur maximum de $t1$ est écrasée par 0, et elle n'apparait donc pas dans le résultat du *max-unpooling*. Ce qui ne sera pas le cas où l'on surveille le neurone $(0, 1)$ et que l'on applique le "nettoyage" de $t2$.